# PEFT Trained model demonstration

- ran in: Google Colab Tesla T4 GPU in VS Code extension

In [1]:
# installing dependincies
%pip install -q -U transformers peft bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 102.4 MB/s eta 0:00:0000:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.2 MB/s eta 0:00:00:00:0100:01


In [2]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:134: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


## Model and PEFT parameters loading

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# Configuration IDs
base_model_id = "Qwen/Qwen2.5-Math-7B"
peft_model_id = "thakur-tejasva/Qwen2.5-Math-7B-PEFT-SOC-Project"

# Configure VRAM footprint for T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("Loading base model in 4-bit...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

print("Loading your custom mathematical PEFT adapters...")
model = PeftModel.from_pretrained(base_model, peft_model_id)

tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True)
print("Model ready for evaluation!")

Loading base model in 4-bit...


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.32k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Model ready for evaluation!


In [4]:
# Cell 4
from IPython.display import display, Markdown

# Ask a question directly inside the notebook cell interface
math_query = "Evaluate the triple integral $$\iiint_E z \, dV$$, where $E$ is the solid region in the first octant bounded by the sphere $x^2 + y^2 + z^2 = 4$ and the coordinate planes. Provide step-by-step mathematical reasoning using LaTeX."

# Match the exact ChatML formatting template the model trained on
prompt = f"<|im_start|>user\n{math_query}<|im_end|>\n<|im_start|>assistant\n"

# Tokenize and transfer tensors to GPU
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Generate output tokens
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=1024,
        do_sample=False, # Use greedy decoding for precise deterministic math execution
        repetition_penalty=1.1
    )

# Slice out the prompt sequence to only display the solution text
generated_tokens = output_ids[0][len(inputs.input_ids[0]):]
solution = tokenizer.decode(generated_tokens, skip_special_tokens=True)

# Markdown parsing renders LaTeX elements instantly inside your notebook
display(Markdown(f"### Query:\n{math_query}\n\n"))
display(Markdown(f"### Fine-Tuned Qwen Output:\n\n{solution}"))

<>:5: SyntaxWarning: invalid escape sequence '\i'
<>:5: SyntaxWarning: invalid escape sequence '\i'
/tmp/ipykernel_797/1290652527.py:5: SyntaxWarning: invalid escape sequence '\i'
  math_query = "Evaluate the triple integral $$\iiint_E z \, dV$$, where $E$ is the solid region in the first octant bounded by the sphere $x^2 + y^2 + z^2 = 4$ and the coordinate planes. Provide step-by-step mathematical reasoning using LaTeX."
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


### Query:
Evaluate the triple integral $$\iiint_E z \, dV$$, where $E$ is the solid region in the first octant bounded by the sphere $x^2 + y^2 + z^2 = 4$ and the coordinate planes. Provide step-by-step mathematical reasoning using LaTeX.



### Fine-Tuned Qwen Output:

To evaluate the given triple integral, we will use spherical coordinates.
In spherical coordinates, a point $(x,y,z)$ can be represented as $(r,\theta,\phi)$,
where $r$ is the distance from the origin to the point,
$\theta$ is the angle between the positive x-axis and the projection of the point onto the xy-plane,
and $\phi$ is the angle between the positive z-axis and the line connecting the origin to the point.

The bounds for $r$, $\theta$, and $\phi$ are determined by the region $E$.
Since $E$ is in the first octant, we have $0 \leq r \leq 2$ (the radius of the sphere),
$0 \leq \theta \leq \frac{\pi}{2}$ (since we're only considering the first quadrant in the xy-plane),
and $0 \leq \phi \leq \frac{\pi}{2}$ (since we're only considering the upper half of the sphere).

Now, let's set up the integral:
$$\iiint_E z \, dV = \int_0^{\frac{\pi}{2}} \int_0^{\frac{\pi}{2}} \int_0^2 (r\cos(\phi)) \cdot r^2 \sin(\phi) \, dr \, d\theta \, d\phi.$$

We integrate with respect to $r$ first:
$$\int_0^2 r^3 \sin(\phi)\cos(\phi) \, dr = \left[\frac{1}{4}r^4 \sin(2\phi)\right]_0^2 = \frac{1}{4}(2)^4 \sin(2\phi) - \frac{1}{4}(0)^4 \sin(2\phi) = 4\sin(2\phi).$$

Next, we integrate with respect to $\theta$:
$$\int_0^{\frac{\pi}{2}} 4\sin(2\phi) \, d\theta = [4(-\cos(2\phi))]_0^{\frac{\pi}{2}} = 4(-\cos(2\cdot\frac{\pi}{2})) - 4(-\cos(2\cdot0)) = 8.$$

Finally, we integrate with respect to $\phi$:
$$\int_0^{\frac{\pi}{2}} 8 \, d\phi = [8\phi]_0^{\frac{\pi}{2}} = 8\left(\frac{\pi}{2}\right) - 8(0) = 4\pi.$$

Therefore, the value of the triple integral is $\boxed{4\pi}$.
The answer is: 4\pi�